In [1]:
# ============================================================
# 1. Install packages, import libraries, initialize settings
# ============================================================

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

get_ipython().system('pip -q install git+https://github.com/sokrypton/ColabDesign.git@v1.1.1')
get_ipython().system('pip -q uninstall -y ibis-framework')
get_ipython().system('pip -q install -U "toolz>=1.0.0"')

import re, gc, json, gzip, zipfile, traceback, random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.special import softmax

from google.colab import drive
from tqdm.notebook import tqdm

from colabdesign.mpnn import mk_mpnn_model
from colabdesign.mpnn.model import residue_constants
from Bio.PDB import MMCIFParser, PDBIO, Select
import jax

print("All packages imported.")
print("AA order:", "".join(residue_constants.restypes))


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.3/374.3 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.1.0 which is incompatible.
All packages imported.
AA order: ARNDCQEGHILKMFPSTWYV


In [2]:
# ============================================================
# 2. Mount Google Drive
# ============================================================

try:
    drive.flush_and_unmount()
    print("Old Drive mount flushed.")
except Exception:
    print("No previous Drive mount to flush.")

drive.mount("/content/drive", force_remount=True)

DRIVE_ROOT = Path("/content/drive/MyDrive")
if DRIVE_ROOT.exists():
    print("Drive mounted:", DRIVE_ROOT)
else:
    raise RuntimeError("Drive mount failed.")

Drive not mounted, so nothing to flush and unmount.
Old Drive mount flushed.
Mounted at /content/drive
Drive mounted: /content/drive/MyDrive


In [27]:
# ============================================================
# 3. Project paths
# ============================================================

# 基础目录: /content/drive/MyDrive/Colab Notebooks/test/DMS/
BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/test/DMS")

# 输入路径: {BASE_DIR}/AlphaFold/蛋白ID/pdb文件
# 修正为 AlphaFold (F大写)
INPUT_DIR = BASE_DIR / "AlphaFold"

# 输出保存路径: {BASE_DIR}/MPNN/蛋白ID/
PROJECT_DIR = BASE_DIR / "MPNN"
MODE_DIR = PROJECT_DIR

TMP_ROOT = Path("/content/tmp_mpnn_run")
TARGET_CHAIN = "A"

for d in [PROJECT_DIR, TMP_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print("Current Configuration:")
print(f"- Input Root  : {INPUT_DIR}")
print(f"- Output Root : {PROJECT_DIR}")
print(f"- Files will be at: {PROJECT_DIR}/[Protein_ID]/")

# 检查路径是否存在
if not INPUT_DIR.exists():
    print(f"⚠️ 警告: 找不到输入目录 {INPUT_DIR}")
    print("提示: 请确保 Drive 中 'DMS' 下的文件夹名确实是 'AlphaFold' (注意 F 的大小写)。")
else:
    print("✅ 输入目录已找到！")

Current Configuration:
- Input Root  : /content/drive/MyDrive/Colab Notebooks/test/DMS/AlphaFold
- Output Root : /content/drive/MyDrive/Colab Notebooks/test/DMS/MPNN
- Files will be at: /content/drive/MyDrive/Colab Notebooks/test/DMS/MPNN/[Protein_ID]/
✅ 输入目录已找到！


In [36]:
# ============================================================
# 4. Runtime configuration  (edit these before running)
# ============================================================

RANDOM_SEED = 45
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
jax_rng_key = jax.random.PRNGKey(RANDOM_SEED)
print(f"Seeds fixed: Python/NumPy={RANDOM_SEED}, JAX key initialised.")

MODEL_NAME = "v_48_020"
RESPONSE_DEFINITION = "max_delta_probability_energy_gap_from_masked_probability_tensor"

MAX_PROTEINS   = None
MAX_LENGTH     = None
SKIP_FINISHED  = False
RESUME_PARTIAL = False
SAVE_EVERY_POS = 10

EPS_PROB = 1e-9

AA_LIST      = list(residue_constants.restypes)
TARGET_CHAIN = "A"

THREE_TO_ONE = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C",
    "GLN":"Q","GLU":"E","GLY":"G","HIS":"H","ILE":"I",
    "LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P",
    "SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M",
}

print("Mode         : xscan tensor generation only")
print(f"R definition : {RESPONSE_DEFINITION}")
print(f"MODEL_NAME   : {MODEL_NAME}")
print(f"MAX_PROTEINS : {MAX_PROTEINS}")
print(f"INPUT_DIR    : {INPUT_DIR}")


Seeds fixed: Python/NumPy=45, JAX key initialised.
Mode         : xscan tensor generation only
R definition : max_delta_probability_energy_gap_from_masked_probability_tensor
MODEL_NAME   : v_48_020
MAX_PROTEINS : None
INPUT_DIR    : /content/drive/MyDrive/Colab Notebooks/test/DMS/AlphaFold


In [37]:
# ============================================================
# 5. Initialize ProteinMPNN
# ============================================================

mpnn_model = mk_mpnn_model(MODEL_NAME)
print(f"ProteinMPNN loaded: {MODEL_NAME}")

ProteinMPNN loaded: v_48_020


In [38]:
# ============================================================
# 6. File discovery and CIF/ZIP handling
# ============================================================

def get_accession_from_filename(filepath):
    """
    Extracts the Protein ID based on the structure:
    /AlphaFold/{Protein_ID}/pdb文件/{filename}.pdb
    """
    p = Path(filepath)
    path_parts = p.parts

    # 查找 'AlphaFold' 文件夹后的第一个文件夹名作为 ID
    if "AlphaFold" in path_parts:
        idx = path_parts.index("AlphaFold")
        if idx + 1 < len(path_parts):
            return path_parts[idx + 1]

    # 如果结构不同，尝试获取文件的直接上级或父级的父级
    if p.parent.name == "pdb文件":
        return p.parent.parent.name

    return p.stem

def list_protein_files(input_dir):
    found = []
    # 搜索结构: AlphaFold/蛋白ID/pdb文件/*.pdb
    patterns = ["**/pdb文件/*.pdb", "**/pdb文件/*.pdb.gz", "**/pdb文件/*.zip"]

    for pattern in patterns:
        for p in sorted(input_dir.rglob(pattern)):
            if p.is_file():
                found.append(p)

    if not found:
        for p in sorted(input_dir.rglob("*.pdb")):
            if p.is_file():
                found.append(p)

    return sorted(list(set(found)))

# ... (Rest of the utility functions) ...

class _ProteinChainSelect(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id
    def accept_chain(self, chain):
        return chain.id == self.chain_id
    def accept_residue(self, residue):
        return residue.id[0] == " "

def convert_cif_to_pdb(cif_path, out_pdb_path, target_chain="A"):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("af3", str(cif_path))
    model = structure[0]
    chain_ids = [c.id for c in model.get_chains()]
    if target_chain in chain_ids:
        use_chain = target_chain
    elif chain_ids:
        use_chain = chain_ids[0]
        model[use_chain].id = target_chain
    else:
        raise ValueError(f"No chains in {cif_path}")
    io = PDBIO()
    io.set_structure(model)
    io.save(str(out_pdb_path), select=_ProteinChainSelect(target_chain))

def _find_sequence_in_json(data):
    if isinstance(data, dict):
        if "sequence" in data and isinstance(data["sequence"], str):
            seq = data["sequence"]
            if len(seq) > 5 and all(c in "ARNDCQEGHILKMFPSTWYVX" for c in seq.upper()):
                return seq
        for v in data.values():
            r = _find_sequence_in_json(v)
            if r: return r
    if isinstance(data, list):
        for item in data:
            r = _find_sequence_in_json(item)
            if r: return r
    return None

def extract_zip_protein(zip_path, tmp_root, target_chain="A"):
    tmp_root.mkdir(parents=True, exist_ok=True)
    accession = zip_path.stem
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        cif_name  = next((n for n in names if n.endswith("model_0.cif")), None)
        json_name = next((n for n in names if n.lower().endswith("job_request.json")), None)
        if not cif_name or not json_name:
            raise FileNotFoundError(f"Missing files in {zip_path.name}")
        tmp_cif = tmp_root / f"{accession}_model_0.cif"
        tmp_cif.write_bytes(zf.read(cif_name))
        tmp_pdb = tmp_root / f"{accession}.pdb"
        convert_cif_to_pdb(tmp_cif, tmp_pdb, target_chain=target_chain)
        tmp_cif.unlink()
        job_data = json.loads(zf.read(json_name))
        sequence = _find_sequence_in_json(job_data)
    return tmp_pdb, sequence, accession

print("File utilities corrected: ID extraction now targets the {ProteinID} folder.")

File utilities corrected: ID extraction now targets the {ProteinID} folder.


In [39]:
# ============================================================
# 7. PDB sequence reader
# ============================================================


def read_pdb_sequence(pdb_file, chain_id="A"):
    residues, seen = [], set()
    with open(pdb_file) as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue
            if line[12:16].strip() != "CA":
                continue
            if line[21].strip() != chain_id:
                continue
            key = (line[21], line[22:26].strip(), line[26].strip())
            if key in seen:
                continue
            seen.add(key)
            residues.append(THREE_TO_ONE.get(line[17:20].strip(), "X"))
    seq = "".join(residues)
    if not seq:
        raise ValueError(f"No sequence for chain {chain_id} in {pdb_file}")
    return seq


def clean_sequence_for_mpnn(seq):
    invalid = sorted(set(seq) - set(AA_LIST))
    if invalid:
        raise ValueError(f"Unsupported residues: {invalid}")
    return seq

In [40]:
# ============================================================
# 8. ProteinMPNN probability extraction
# ============================================================


def get_mpnn_probs(mpnn_model, seq, mode="conditional"):
    L = sum(mpnn_model._lengths)
    if mode == "unconditional":
        ar_mask = np.zeros((L, L), dtype=np.float32)
    elif mode == "conditional":
        ar_mask = 1 - np.eye(L, dtype=np.float32)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    result = mpnn_model.score(seq=seq, ar_mask=ar_mask)
    logits = np.asarray(result["logits"]).squeeze()
    if logits.ndim == 3:
        logits = logits[0]
    if logits.shape[0] != L and logits.shape[1] == L:
        logits = logits.T
    if logits.shape[0] != L:
        raise ValueError(f"Logits shape mismatch: {logits.shape}, L={L}")
    return softmax(logits[:, :20], axis=-1).astype(np.float32)

In [41]:
# ============================================================
# 9. Core tensor generation functions
# ============================================================


def get_masked_source_probs(mpnn_model, wt_seq, source_j):
    """
    Return P_j_rm, an L x 20 probability matrix after masking source position j.
    This is the same masked forward pass used by the original X-scan routine.
    """
    L = len(wt_seq)
    ar_mask_j = 1 - np.eye(L, dtype=np.float32)
    ar_mask_j[:, source_j] = 0.0

    result = mpnn_model.score(seq=wt_seq, ar_mask=ar_mask_j)
    logits = np.asarray(result["logits"]).squeeze()
    if logits.ndim == 3:
        logits = logits[0]
    if logits.shape[0] != L and logits.shape[1] == L:
        logits = logits.T
    if logits.shape[0] != L:
        raise ValueError(f"Logits shape mismatch: {logits.shape}, L={L}")
    return softmax(logits[:, :20], axis=-1).astype(np.float32)


def compute_xscan_probability_tensor(mpnn_model, wt_seq, save_every=10):
    """
    Build d_tensor[i, j, a] = P_j_rm[i, a].

    Axis convention:
      i: receiver residue position
      j: masked source residue position
      a: amino-acid index in AA_LIST
    """
    L = len(wt_seq)
    d_tensor = np.zeros((L, L, 20), dtype=np.float32)

    with tqdm(total=L, desc="X-scan probability tensor", leave=False) as pbar:
        for j in range(L):
            d_tensor[:, j, :] = get_masked_source_probs(mpnn_model, wt_seq, j)
            pbar.update(1)
            if save_every and (j + 1) % save_every == 0:
                print(f"  computed masked source positions: {j + 1}/{L}")

    return d_tensor


In [42]:
# ============================================================
# 10. Single-protein xscan tensor runner
# ============================================================

def run_one_protein_response_matrix(
    protein_name, pdb_file, chain_id, output_dir, mpnn_model,
    override_seq=None, skip_finished=True, resume_partial=True,
    max_length=None, save_every=10,
):
    # 更新路径逻辑: {output_dir}/{protein_name}/{file_name}
    protein_dir = output_dir / protein_name
    protein_dir.mkdir(parents=True, exist_ok=True)

    metadata_path = protein_dir / f"{protein_name}_metadata.json"
    npz_path      = protein_dir / f"{protein_name}_xscan_probability_tensors.npz"
    required = [metadata_path, npz_path]

    if skip_finished and all(p.exists() for p in required):
        print(f"  [SKIP] {protein_name}: outputs exist.")
        return {"protein": protein_name, "status": "skipped_existing", "error": ""}

    # Sequence
    if override_seq is not None:
        wt_seq = clean_sequence_for_mpnn(override_seq)
        print(f"  Sequence from JSON: {len(wt_seq)} residues")
    else:
        wt_seq = clean_sequence_for_mpnn(read_pdb_sequence(pdb_file, chain_id))

    L = len(wt_seq)
    print(f"  Length: {L}  |  Mode: xscan tensor generation")
    if max_length and L > max_length:
        print(f"  [SKIP] L={L} > MAX_LENGTH={max_length}")
        return {"protein": protein_name, "status": "skipped_too_long", "length": L, "error": ""}

    wt_indices = np.asarray([AA_LIST.index(aa) for aa in wt_seq], dtype=np.int32)

    # Load structure
    mpnn_model.prep_inputs(
        pdb_filename=str(pdb_file), chain=chain_id,
        homooligomer=False, fix_pos=None, inverse=True, verbose=False,
    )

    print("  Computing P_wt conditional baseline ...", end=" ", flush=True)
    P_wt = get_mpnn_probs(mpnn_model, wt_seq, mode="conditional")
    print("done.")

    print("  Computing P_uncond unconditional baseline ...", end=" ", flush=True)
    P_uncond = get_mpnn_probs(mpnn_model, wt_seq, mode="unconditional")
    print("done.")

    print(f"  Computing d_tensor_LxLx20 from {L} masked source positions ...")
    d_tensor = compute_xscan_probability_tensor(
        mpnn_model=mpnn_model,
        wt_seq=wt_seq,
        save_every=save_every,
    )

    np.savez_compressed(
        npz_path,
        P_wt_Lx20=P_wt,
        P_uncond_Lx20=P_uncond,
        d_tensor_LxLx20=d_tensor,
        wt_indices=wt_indices,
        wt_sequence=np.asarray(wt_seq),
        aa_order=np.asarray("".join(AA_LIST)),
    )

    meta = {
        "protein": protein_name,
        "chain_id": chain_id,
        "length": L,
        "wt_sequence": wt_seq,
        "model_name": MODEL_NAME,
        "mode": "xscan_probability_tensor_generation",
        "response_definition": RESPONSE_DEFINITION,
        "random_seed": RANDOM_SEED,
        "aa_order": "".join(AA_LIST),
        "eps_prob": EPS_PROB,
        "n_forward_passes": L + 2,
        "tensor_definition": "d_tensor_LxLx20[i, j, a] = P_j_rm[i, a], where j is the masked source position",
        "npz_file": str(npz_path),
        "npz_arrays": ["P_wt_Lx20", "P_uncond_Lx20", "d_tensor_LxLx20", "wt_indices", "wt_sequence", "aa_order"],
        "timestamp_utc": datetime.utcnow().isoformat(),
    }
    with open(metadata_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"  Saved metadata: {metadata_path.name}")
    print(f"  Saved tensors : {npz_path.name}")
    print(f"  [DONE] {protein_name}  ({L + 2} forward passes)")

    return {
        "protein": protein_name,
        "status": "finished",
        "length": L,
        "n_forward_passes": L + 2,
        "metadata_path": str(metadata_path),
        "npz_path": str(npz_path),
        "error": "",
    }

In [43]:
# ============================================================
# 11. Batch scan proteins from INPUT_DIR
# ============================================================

if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input directory not found: {INPUT_DIR}")

protein_files  = list_protein_files(INPUT_DIR)
members_to_run = protein_files if MAX_PROTEINS is None else protein_files[:MAX_PROTEINS]

print(f"Found {len(protein_files)} protein files in '{INPUT_DIR.name}'.")
print(f"Will process {len(members_to_run)} proteins.")
print(f"Output: {MODE_DIR}")

batch_logs = []

for idx, fpath in enumerate(tqdm(members_to_run, desc="Proteins"), start=1):
    accession = get_accession_from_filename(fpath)
    file_type = ("zip" if fpath.name.lower().endswith(".zip") else
                 "pdb.gz" if fpath.name.lower().endswith(".pdb.gz") else "pdb")
    print(f"\n[{idx}/{len(members_to_run)}] {accession}  ({file_type})")

    tmp_pdb, override_seq = None, None

    try:
        if file_type == "zip":
            tmp_pdb, override_seq, accession = extract_zip_protein(
                fpath, TMP_ROOT, target_chain=TARGET_CHAIN)
        elif file_type == "pdb.gz":
            tmp_pdb = TMP_ROOT / f"{accession}.pdb"
            tmp_pdb.write_bytes(gzip.decompress(fpath.read_bytes()))
        else:
            tmp_pdb = fpath

        log = run_one_protein_response_matrix(
            protein_name=accession, pdb_file=tmp_pdb, chain_id=TARGET_CHAIN,
            output_dir=MODE_DIR, mpnn_model=mpnn_model, override_seq=override_seq,
            skip_finished=SKIP_FINISHED, resume_partial=RESUME_PARTIAL,
            max_length=MAX_LENGTH, save_every=SAVE_EVERY_POS)
        batch_logs.append(log)

    except Exception as e:
        print(f"  [ERROR] {accession}: {e}")
        traceback.print_exc()
        batch_logs.append({"protein": accession, "status": "failed", "error": str(e)})

    finally:
        if tmp_pdb is not None and tmp_pdb != fpath and tmp_pdb.exists():
            tmp_pdb.unlink()
        gc.collect()

log_df = pd.DataFrame(batch_logs)
print("\nBatch finished.")
display(log_df)


Found 3 protein files in 'AlphaFold'.
Will process 3 proteins.
Output: /content/drive/MyDrive/Colab Notebooks/test/DMS/MPNN


Proteins:   0%|          | 0/3 [00:00<?, ?it/s]


[1/3] 00000037-a  (pdb)
  Length: 381  |  Mode: xscan tensor generation
  Computing P_wt conditional baseline ... done.
  Computing P_uncond unconditional baseline ... done.
  Computing d_tensor_LxLx20 from 381 masked source positions ...


X-scan probability tensor:   0%|          | 0/381 [00:00<?, ?it/s]

  computed masked source positions: 10/381
  computed masked source positions: 20/381
  computed masked source positions: 30/381
  computed masked source positions: 40/381
  computed masked source positions: 50/381
  computed masked source positions: 60/381
  computed masked source positions: 70/381
  computed masked source positions: 80/381
  computed masked source positions: 90/381
  computed masked source positions: 100/381
  computed masked source positions: 110/381
  computed masked source positions: 120/381
  computed masked source positions: 130/381
  computed masked source positions: 140/381
  computed masked source positions: 150/381
  computed masked source positions: 160/381
  computed masked source positions: 170/381
  computed masked source positions: 180/381
  computed masked source positions: 190/381
  computed masked source positions: 200/381
  computed masked source positions: 210/381
  computed masked source positions: 220/381
  computed masked source positions: 230/3

...

X-scan probability tensor:   0%|          | 0/189 [00:00<?, ?it/s]

  computed masked source positions: 10/189
  computed masked source positions: 20/189
  computed masked source positions: 30/189
  computed masked source positions: 40/189
  computed masked source positions: 50/189
  computed masked source positions: 60/189
  computed masked source positions: 70/189
  computed masked source positions: 80/189
  computed masked source positions: 90/189
  computed masked source positions: 100/189
  computed masked source positions: 110/189
  computed masked source positions: 120/189
  computed masked source positions: 130/189
  computed masked source positions: 140/189
  computed masked source positions: 150/189
  computed masked source positions: 160/189
  computed masked source positions: 170/189
  computed masked source positions: 180/189


/tmp/ipykernel_1027/1082875387.py:83: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat(),


  Saved metadata: 00000115-a_metadata.json
  Saved tensors : 00000115-a_xscan_probability_tensors.npz
  [DONE] 00000115-a  (191 forward passes)

Batch finished.


,protein,status,length,n_forward_passes,metadata_path,npz_path,error
0,00000037-a,finished,381,383,/content/drive/MyDrive/Colab Notebooks/test/DM...,/content/drive/MyDrive/Colab Notebooks/test/DM...,
1,00000080-a,finished,238,240,/content/drive/MyDrive/Colab Notebooks/test/DM...,/content/drive/MyDrive/Colab Notebooks/test/DM...,
2,00000115-a,finished,189,191,/content/drive/MyDrive/Colab Notebooks/test/DM...,/content/drive/MyDrive/Colab Notebooks/test/DM...,


In [12]:
# ============================================================
# 15. Inspect a single protein's outputs
# ============================================================

INSPECT_PROTEIN = None

result_dirs = [d for d in sorted(MODE_DIR.iterdir()) if d.is_dir()]

if not result_dirs:
    print("No result directories found.")
else:
    target_dir = (MODE_DIR / INSPECT_PROTEIN) if INSPECT_PROTEIN else result_dirs[0]
    pname = target_dir.name
    print(f"Inspecting: {pname}")

    with open(target_dir / f"{pname}_metadata.json") as f:
        meta = json.load(f)
    print(json.dumps(meta, indent=2))

    npz_path = target_dir / f"{pname}_xscan_probability_tensors.npz"
    with np.load(npz_path) as data:
        for key in data.files:
            arr = data[key]
            print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")


No result directories found.


In [ ]:
# ============================================================
# 16. List all output files
# ============================================================

print("Output root:", MODE_DIR)
for d in sorted(MODE_DIR.iterdir()):
    if not d.is_dir():
        continue
    print(f"\n  [{d.name}]")
    for f in sorted(d.rglob("*")):
        if f.is_file():
            size_kb = f.stat().st_size / 1024
            print(f"    {str(f.relative_to(d)):<55s}  {size_kb:>8.1f} KB")
